# Étape 1 : le poids des mines dans l'économie guinéenne

**Question de l'étape :** depuis quand la Guinée dépend-elle autant des mines, et plus ou moins que ses voisins ?

**Ce que vous allez apprendre :**
- télécharger des données publiques depuis Python ;
- les explorer **en SQL** avec DuckDB, votre point fort ;
- passer en **pandas** pour faire votre premier graphique.

**Avant de commencer :** placez ce notebook dans `guinee-mines-data/notebooks/`, puis installez les outils une fois, dans un terminal VS Code :
```
pip install duckdb pandas matplotlib
```

## Partie A : télécharger les données (fourni, à exécuter)

Cette cellule interroge la Banque mondiale et crée 3 fichiers CSV dans un dossier `data/`. Vous n'avez rien à écrire, mais **lisez-la** : vous y reconnaîtrez un `with open(..., "w")`, une boucle et un dictionnaire, comme dans vos exercices.

Les 3 indicateurs :
| Fichier | Indicateur | Code Banque mondiale |
|---|---|---|
| `rente_miniere.csv` | Rente minière, en % du PIB | NY.GDP.MINR.RT.ZS |
| `export_minerais.csv` | Minerais et métaux, en % des exportations de marchandises | TX.VAL.MMTL.ZS.UN |
| `pib.csv` | PIB, en dollars courants | NY.GDP.MKTP.CD |

Si la cellule affiche une erreur, copiez-moi la dernière ligne du message.

In [ ]:
import json
import csv
import os
import urllib.request

PAYS = "GIN;SLE;MLI;CIV;SEN"   # Guinée, Sierra Leone, Mali, Côte d'Ivoire, Sénégal

def telecharger_indicateur(code, nom_fichier):
    """Télécharge un indicateur de la Banque mondiale et l'enregistre en CSV dans data/."""
    url = (f"https://api.worldbank.org/v2/country/{PAYS}/indicator/{code}"
           f"?format=json&per_page=2000&date=2000:2024")
    with urllib.request.urlopen(url) as reponse:
        contenu = json.load(reponse)
    lignes = contenu[1]          # contenu[0] = infos sur la requête, contenu[1] = les données

    os.makedirs("data", exist_ok=True)
    with open(f"data/{nom_fichier}", "w", encoding="utf-8", newline="") as f:
        ecrivain = csv.writer(f)
        ecrivain.writerow(["pays", "code_pays", "annee", "valeur"])
        for l in lignes:
            ecrivain.writerow([l["country"]["value"], l["countryiso3code"], l["date"], l["value"]])
    print(f"{nom_fichier} : {len(lignes)} lignes enregistrées")

telecharger_indicateur("NY.GDP.MINR.RT.ZS", "rente_miniere.csv")      # rente minière, en % du PIB
telecharger_indicateur("TX.VAL.MMTL.ZS.UN", "export_minerais.csv")    # minerais et métaux, en % des exportations
telecharger_indicateur("NY.GDP.MKTP.CD", "pib.csv")                   # PIB, en dollars courants

## Partie B : explorer en SQL (à vous)

DuckDB permet d'écrire du SQL **directement sur un fichier CSV**, comme si c'était une table. La syntaxe :
```python
duckdb.sql("VOTRE REQUÊTE ICI").show()
```
Le nom de la table, c'est le chemin du fichier entre apostrophes : `FROM 'data/rente_miniere.csv'`.

Une valeur vide dans le fichier devient `NULL`, comme dans une base de données.

In [ ]:
import duckdb

# Exemple : les 5 premières lignes
duckdb.sql("SELECT * FROM 'data/rente_miniere.csv' LIMIT 5").show()

### B1. Combien de lignes par pays dans `rente_miniere.csv` ?

In [ ]:
# Votre requête ici


### B2. Combien de valeurs manquantes (`valeur IS NULL`) par pays ?
Notez-le : une donnée manquante est une information en soi.

In [ ]:
# Votre requête ici


### B3. La rente minière de la Guinée, année par année, de la plus récente à la plus ancienne

In [ ]:
# Votre requête ici


### B4. La rente minière **moyenne** de chaque pays depuis 2010, du plus dépendant au moins dépendant
Arrondissez à 1 décimale avec `ROUND(..., 1)`.

In [ ]:
# Votre requête ici


### B5. L'année où la rente minière de la Guinée a été la plus forte, avec sa valeur

In [ ]:
# Votre requête ici


### B6. Même question que B4, mais pour la part des minerais dans les exportations (`export_minerais.csv`)

In [ ]:
# Votre requête ici


<details><summary>Indices pour la partie B</summary>

- B1 : `GROUP BY pays` avec `COUNT(*)`.
- B2 : `COUNT(*)` avec `WHERE valeur IS NULL`, ou `SUM(CASE WHEN valeur IS NULL THEN 1 ELSE 0 END)`.
- B3 : `WHERE code_pays = 'GIN'` et `ORDER BY annee DESC`.
- B4 : `WHERE annee >= 2010`, `AVG(valeur)`, `GROUP BY pays`, `ORDER BY ... DESC`.
- B5 : `ORDER BY valeur DESC LIMIT 1`, sans oublier d'écarter les `NULL`.

</details>

## Partie C : votre premier graphique avec pandas

Pour faire des graphiques, on passe le résultat SQL dans un **DataFrame** pandas : c'est un tableau, l'équivalent Python d'une table SQL. `.df()` fait la conversion.

Ensuite, `pivot` met une colonne par pays, ce qui permet de tracer une courbe par pays.

In [ ]:
import matplotlib.pyplot as plt

rente = duckdb.sql("SELECT pays, annee, valeur FROM 'data/rente_miniere.csv'").df()
print(rente.head())        # les 5 premières lignes du DataFrame

tableau = rente.pivot(index="annee", columns="pays", values="valeur")
print(tableau.tail())      # les 5 dernières années, une colonne par pays

In [ ]:
tableau.plot(figsize=(10, 5))
plt.title("Rente minière en % du PIB")
plt.xlabel("Année")
plt.ylabel("% du PIB")
plt.grid(True, alpha=0.3)

os.makedirs("images", exist_ok=True)
plt.savefig("images/rente_miniere.png", dpi=150, bbox_inches="tight")
plt.show()

### C1. À vous : faites le même graphique pour `export_minerais.csv`
Copiez les deux cellules au-dessus et adaptez-les : le fichier, le titre, l'axe y et le nom de l'image.

In [ ]:
# Votre code ici


## Partie D : vos constats

Écrivez **3 phrases** de conclusion, chacune avec un chiffre tiré de vos requêtes. Exemple de forme : « Depuis 2010, la rente minière représente en moyenne X % du PIB guinéen, contre Y % au Mali. »

Ces phrases iront directement dans le README du projet.

1. ...
2. ...
3. ...

## Pour finir

1. Enregistrez (Ctrl + S).
2. Faites un commit sur GitHub : `git add .`, puis `git commit -m "Étape 1 : poids des mines dans l'économie"`, puis `git push`.
3. Envoyez-moi le notebook dans le fil.